# P1 — Exact Straight-Segment Geometry Feasibility


        P1은 edge 위 몇 점만 검사하던 방식을 바꿔, **edge 전체에서 terrain과
        LOS를 실제로 침범하지 않는지** 확인한다. Cubic terrain은 knot와
        stationary point를, piecewise-linear LOS는 모든 breakpoint를 검사한다.

In [1]:
from pathlib import Path
import json
from html import escape
import subprocess
import sys
from IPython.display import display, Markdown, Image, HTML, FileLink

def locate_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "p1b_4D").is_dir() and (candidate / "p1b_roadmap_0729.md").exists():
            return candidate
    raise RuntimeError("glider_hybrid_control repository root를 찾지 못했습니다.")

ROOT = locate_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"
STAGE = "P1"
STATUS = "COMPLETED"

def run_module(module, *arguments, timeout=None):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    completed = subprocess.run(
        command, cwd=ROOT, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=timeout,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"{module} 실행 실패: exit={completed.returncode}")
    return completed.stdout

def load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"결과 파일이 없습니다: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def show_png(path, width=1050):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"> ⚠️ 그림이 없습니다: `{path}`"))

def display_table(rows, columns=None):
    if isinstance(rows, dict):
        rows = [rows]
    rows = list(rows)
    if columns is None:
        columns = []
        for row in rows:
            for key in row:
                if key not in columns:
                    columns.append(key)
    if not rows:
        display(Markdown("_(표시할 행이 없습니다.)_"))
        return
    def cell(value):
        if isinstance(value, float):
            value = f"{value:.8g}"
        elif isinstance(value, (dict, list, tuple)):
            value = json.dumps(value, ensure_ascii=False)
        return escape(str(value))
    header = "".join(f"<th>{cell(name)}</th>" for name in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{cell(row.get(name, ''))}</td>" for name in columns) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<div style='overflow-x:auto'><table>"
        f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"
    ))

display(Markdown(f"**{STAGE} 상태:** `{STATUS}`  \nRepository: `{ROOT}`"))

**P1 상태:** `COMPLETED`  
Repository: `C:\Users\jeffe\Desktop\git\glider_hybrid_control`

In [2]:
RERUN_AUDIT = True
RUN_REGRESSION_TESTS = True
RUN_FULL_SUITE = False  # True이면 약 7분 이상 걸릴 수 있습니다.
AUDIT_PATH = RESULTS / "direction_b" / "p1_exact_geometry_audit.json"

if RERUN_AUDIT:
    run_module(
        "p1b_4D.audit_p1_direction_b_geometry_certificates",
        "--project-root", ROOT, "--output", AUDIT_PATH,
    )
audit = load_json(AUDIT_PATH)
display_table(audit["summary"])

2026-07-30 09:45:03,578 | INFO | stackelberg | phase=Phase 1: Configuration status=started
2026-07-30 09:45:03,578 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehicle segment_length is deferred to the future transcription contract instead of fabricating a Phase 1 value
2026-07-30 09:45:03,578 | INFO | stackelberg | phase=Phase 1: Configuration status=success elapsed_seconds=0.000371
{
  "stored_case_count": 54,
  "selected_policy_count": 42,
  "not_applicable_infeasible_case_count": 12,
  "certified_policy_count": 42,
  "failed_policy_count": 0,
  "all_selected_policies_certified": true
}



stored_case_count,selected_policy_count,not_applicable_infeasible_case_count,certified_policy_count,failed_policy_count,all_selected_policies_certified
54,42,12,42,0,True


In [3]:
applicable = [
    record for record in audit["records"] if record.get("passed") is not None
]
margin_columns = [
    "source", "terrain_name", "sensor_name", "case_id", "passed",
    "minimum_powered_terrain_margin", "minimum_powered_occlusion_margin",
    "minimum_glide_terrain_margin", "minimum_glide_los_margin",
]
display_table(applicable, margin_columns)
numeric_margin_columns = [
    "minimum_powered_terrain_margin", "minimum_powered_occlusion_margin",
    "minimum_glide_terrain_margin", "minimum_glide_los_margin",
]
terrain_minima = []
for terrain_name in sorted({record["terrain_name"] for record in applicable}):
    subset = [record for record in applicable if record["terrain_name"] == terrain_name]
    terrain_minima.append({
        "terrain_name": terrain_name,
        **{
            name: min(record[name] for record in subset)
            for name in numeric_margin_columns
        },
    })
display_table(terrain_minima)

source,terrain_name,sensor_name,case_id,passed,minimum_powered_terrain_margin,minimum_powered_occlusion_margin,minimum_glide_terrain_margin,minimum_glide_los_margin
B2,two_hill,coverage,enriched_v5_q9_l0,True,-1.9287498e-20,3,2.2250694,0
B2,two_hill,coverage,enriched_v5_q9_l1,True,-1.9287498e-20,3,1.123285,0
B2,two_hill,coverage,enriched_v5_q9_l2,True,-1.9287498e-20,3,0.038223445,0
B2,two_hill,coverage,enriched_v9_q9_l1,True,-1.9287498e-20,3,1.123285,0
B2,two_hill,coverage,enriched_v9_q9_l2,True,-1.9287498e-20,3,0.038223445,0
B2,two_hill,coverage,enriched_v5_q17_l2,True,-1.9287498e-20,3,0.038223445,0
B2,two_hill,stackelberg,enriched_v5_q9_l0,True,-1.9287498e-20,3,2.2250694,0
B2,two_hill,stackelberg,enriched_v5_q9_l1,True,-1.9287498e-20,3,0.56286954,0
B2,two_hill,stackelberg,enriched_v5_q9_l2,True,-1.9287498e-20,3,0.13628625,0
B2,two_hill,stackelberg,enriched_v9_q9_l1,True,-1.9287498e-20,3,0.56286954,0


terrain_name,minimum_powered_terrain_margin,minimum_powered_occlusion_margin,minimum_glide_terrain_margin,minimum_glide_los_margin
goal_in_valley,-1.3863433e-47,3,2.2599766,0
single_hill,-6.5874282e-07,3,1.1927184,0
two_hill,-1.9287498e-20,3,0.038223445,0


In [4]:
if RUN_REGRESSION_TESTS:
    run_module(
        "unittest", "p1b_4D.test_segment_feasibility",
        "p1b_4D.test_successor_grid_solver",
    )
    display(Markdown("✅ **P1 exact-geometry와 successor regression 통과**"))
if RUN_FULL_SUITE:
    run_module("unittest", "discover", "-s", "p1b_4D", "-p", "test_*.py")

..2026-07-30 09:45:04,654 | INFO | stackelberg | phase=Phase 1: Configuration status=started
2026-07-30 09:45:04,654 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehicle segment_length is deferred to the future transcription contract instead of fabricating a Phase 1 value
2026-07-30 09:45:04,660 | INFO | stackelberg | phase=Phase 1: Configuration status=success elapsed_seconds=0.000597
.....2026-07-30 09:45:04,684 | INFO | stackelberg | phase=Phase 1: Configuration status=started
2026-07-30 09:45:04,684 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehicle segment_length is deferred to the future transcription contract instead of fabricating a Phase 1 value
2026-07-30 09:45:04,685 | INFO | stackelberg | phase=Phase 1: Configuration status=success elapsed_seconds=0.000246
.2026-07-30 09:45:04,870 | INFO | stackelberg | phase=Phase 1: Configuration status=started
2026-07-30 09:45:04,870 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehi

✅ **P1 exact-geometry와 successor regression 통과**

## 직관적 결론

저장된 B2/B3 사례 54개 중 경로가 존재하는 42개 모두가 exact
geometry certificate를 통과했다. 따라서 P1은 기존 선택 경로와
finite cost를 유지하면서, 다른 sampled-valid edge가 중간에서
장애물을 통과할 가능성을 제거한다.